In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0.0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/ASDIVsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CCoT_prompt_example.txt").read()

In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/ASDIV/CoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth

        prompt_q = (
            CoT_prompt_examples +
            '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accurately."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        # === Determine Correctness
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:06<22:01,  6.48s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 8 / 8 = 100.00%
Accuracy: 8 / 9 = 88.89%
Accuracy: 9 / 10 = 90.00%
Accuracy: 10 / 11 = 90.91%
Accuracy: 11 / 12 = 91.67%
Accuracy: 12 / 13 = 92.31%
Accuracy: 13 / 14 = 92.86%
Accuracy: 14 / 15 = 93.33%
Accuracy: 15 / 16 = 93.75%
Accuracy: 16 / 17 = 94.12%
Accuracy: 17 / 18 = 94.44%
Accuracy: 18 / 19 = 94.74%
Accuracy: 19 / 20 = 95.00%
Accuracy: 20 / 21 = 95.24%
Accuracy: 21 / 22 = 95.45%
Accuracy: 22 / 23 = 95.65%
Accuracy: 23 / 24 = 95.83%
Accuracy: 24 / 25 = 96.00%
Accuracy: 25 / 26 = 96.15%
Accuracy: 25 / 27 = 92.59%
Accuracy: 25 / 28 = 89.29%
Accuracy: 26 / 29 = 89.66%
Accuracy: 26 / 30 = 86.67%
Accuracy: 27 / 31 = 87.10%
Accuracy: 28 / 32 = 87.50%
Accuracy: 29 / 33 = 87.88%
Accuracy: 30 / 34 = 88.24%
Accuracy: 30 / 35 = 85.71%
Accuracy: 31 / 36 = 86.11%
Accuracy: 32 / 37 = 86.49%
Accuracy: 33

 32%|███▏      | 66/205 [00:06<00:10, 13.76it/s]

Accuracy: 57 / 66 = 86.36%
Accuracy: 58 / 67 = 86.57%
Accuracy: 59 / 68 = 86.76%
Accuracy: 60 / 69 = 86.96%
Accuracy: 61 / 70 = 87.14%
Accuracy: 62 / 71 = 87.32%
Accuracy: 63 / 72 = 87.50%


 36%|███▌      | 73/205 [01:01<02:26,  1.11s/it]

...

Accuracy: 160 / 186 = 86.02%
Accuracy: 161 / 187 = 86.10%
Accuracy: 162 / 188 = 86.17%
Accuracy: 163 / 189 = 86.24%
Accuracy: 164 / 190 = 86.32%
Accuracy: 165 / 191 = 86.39%
Accuracy: 166 / 192 = 86.46%
Accuracy: 167 / 193 = 86.53%
Accuracy: 168 / 194 = 86.60%
Accuracy: 168 / 195 = 86.15%
Accuracy: 169 / 196 = 86.22%
Accuracy: 170 / 197 = 86.29%
Accuracy: 170 / 198 = 85.86%
Accuracy: 171 / 199 = 85.93%


 99%|█████████▊| 202/205 [02:07<00:01,  2.57it/s]

Accuracy: 172 / 200 = 86.00%
Accuracy: 173 / 201 = 86.07%
Accuracy: 174 / 202 = 86.14%
Accuracy: 175 / 203 = 86.21%


100%|██████████| 205/205 [02:08<00:00,  1.60it/s]

Accuracy: 175 / 204 = 85.78%
Accuracy: 176 / 205 = 85.85%


In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/ASDIV/standard.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Keep digits, decimal, minus
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth

        prompt_q = (
            Standard_prompt_examples +
            '\nAnswer this question: ' + q + " Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
results = []
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                global acc
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n") 

  0%|          | 1/205 [00:00<00:56,  3.63it/s]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  2%|▏         | 4/205 [00:01<00:58,  3.45it/s]

Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 8 / 8 = 100.00%
Accuracy: 9 / 9 = 100.00%
Accuracy: 10 / 10 = 100.00%
Accuracy: 11 / 11 = 100.00%
Accuracy: 12 / 12 = 100.00%


  6%|▋         | 13/205 [00:03<00:51,  3.69it/s]

...

Accuracy: 146 / 176 = 82.95%
Accuracy: 147 / 177 = 83.05%
Accuracy: 148 / 178 = 83.15%
Accuracy: 149 / 179 = 83.24%
Accuracy: 150 / 180 = 83.33%
Accuracy: 151 / 181 = 83.43%
Accuracy: 152 / 182 = 83.52%
Accuracy: 153 / 183 = 83.61%


 90%|████████▉ | 184/205 [02:04<00:12,  1.66it/s]

Accuracy: 154 / 184 = 83.70%
Accuracy: 155 / 185 = 83.78%
Accuracy: 156 / 186 = 83.87%
Accuracy: 157 / 187 = 83.96%
Accuracy: 158 / 188 = 84.04%
Accuracy: 158 / 189 = 83.60%
Accuracy: 159 / 190 = 83.68%
Accuracy: 160 / 191 = 83.77%
Accuracy: 161 / 192 = 83.85%
Accuracy: 162 / 193 = 83.94%
Accuracy: 162 / 194 = 83.51%
Accuracy: 162 / 195 = 83.08%
Accuracy: 163 / 196 = 83.16%
Accuracy: 164 / 197 = 83.25%


100%|██████████| 205/205 [02:05<00:00,  1.63it/s]

Accuracy: 164 / 198 = 82.83%
Accuracy: 165 / 199 = 82.91%
Accuracy: 166 / 200 = 83.00%
Accuracy: 167 / 201 = 83.08%
Accuracy: 168 / 202 = 83.17%
Accuracy: 169 / 203 = 83.25%
Accuracy: 169 / 204 = 82.84%
Accuracy: 170 / 205 = 82.93%


In [6]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/ASDIV/complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
error_log_path = output_path.replace('.txt', '_errors.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(idx, d):
    global acc, total
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth
        
        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Finish your response with: the answer is <answer>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly.\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Get Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error at index {idx}:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
results = []
start_index = 0  # Start processing from question 127
with open(output_path, 'a') as fd, open(bad_output_path, 'a') as bad_fd, open(error_log_path, 'a') as error_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, idx, d) for idx, d in enumerate(dev_data[start_index:], start=start_index)]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                error_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # Final accuracy report
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n")  # Write to the output file


  0%|          | 1/205 [00:03<10:47,  3.17s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%


  1%|▏         | 3/205 [00:03<03:39,  1.09s/it]

Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 5 / 6 = 83.33%


  3%|▎         | 7/205 [01:57<1:05:03, 19.71s/it]

...

Accuracy: 173 / 202 = 85.64%
Accuracy: 174 / 203 = 85.71%


100%|█████████▉| 204/205 [05:09<00:00,  3.15it/s]

Accuracy: 174 / 204 = 85.29%


100%|██████████| 205/205 [05:09<00:00,  1.51s/it]

Accuracy: 175 / 205 = 85.37%
